<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [ ]:
!pip install torchmetrics

In [ ]:
import math
from functools import partial
from collections import Counter, defaultdict

import io
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.ops import nms, box_iou
from torchvision.models.detection.anchor_utils import AnchorGenerator

from torchmetrics.detection import MeanAveragePrecision

In [ ]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

In [ ]:
import io

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [ ]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

Создаем датасет для предобработки данных

In [ ]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]

        # Получаем данные боксов
        boxes = row['bbox']
        if hasattr(boxes, 'tolist'):
            boxes = boxes.tolist()

        # Если бокс один и это одномерный список [x, y, w, h],
        # оборачиваем его в список списков [[x, y, w, h]]
        if len(boxes) == 4 and isinstance(boxes[0], (int, float)):
            boxes = [boxes]

        # Обязательно конвертируем в numpy array, чтобы избежать ошибки "no attribute 'size'"
        boxes = np.array(boxes, dtype=np.float32)

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [ ]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.RandomResizedCrop(size=(224,224), p=1.0),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    # Раскомментируй, если аугментации изменяют ббоксы.
    # Не забудь указать верный формат для ббоксов.
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'])
)

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)

Не забываем инициализировать наш датасет

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=9, shuffle=False, collate_fn=collate_fn)

In [ ]:
a,b = train_dataset[0]
a.shape

torch.Size([3, 224, 224])

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [ ]:
from torchvision.models import resnet18
import torch.nn.functional as F

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class Backbone(nn.Module):
    def __init__(self, unfreeze_last=1):
        super().__init__()
        # Инициализируем предобученный ResNet18
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)

        # Собираем блоки в список, чтобы удобно управлять заморозкой
        # Исключаем fc слой, так как для детектора он не нужен
        stages = [
            nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool),
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4
        ]

        # 1. Сначала замораживаем абсолютно все параметры во всех блоках
        for stage in stages:
            for param in stage.parameters():
                param.requires_grad = False

        # 2. Размораживаем последние k блоков (например, если unfreeze_last=1, разморозится только layer4)
        if unfreeze_last > 0:
            for stage in stages[-unfreeze_last:]:
                for param in stage.parameters():
                    param.requires_grad = True

        # C3: выход layer2 (stride = 8)
        self.c3_block = nn.Sequential(*stages[:3]) # conv1-maxpool + layer1 + layer2

        # C4: выход layer3 (stride = 16)
        self.c4_block = stages[3] # layer3

        # C5: выход layer4 (stride = 32)
        self.c5_block = stages[4] # layer4

    def forward(self, x):
        c3 = self.c3_block(x)       # [B, 128, H/8, W/8]
        c4 = self.c4_block(c3)      # [B, 256, H/16, W/16]
        c5 = self.c5_block(c4)      # [B, 512, H/32, W/32]
        return (c3, c4, c5)

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Neck(nn.Module):
    def __init__(self, in_ch_list):
        super().__init__()
        i1, i2, i3 = in_ch_list

        # Слой для самого мелкого уровня (карта x1)
        self.con1_1 = nn.Conv2d(i1, 256, kernel_size=1)
        self.con1_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

        # Слой для среднего уровня (карта x2)
        self.con2_1 = nn.Conv2d(i2, 256, kernel_size=1)
        self.con2_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

        # Слой для самого глубокого уровня (карта x3) — исправлены опечатки
        self.con3_1 = nn.Conv2d(i3, 256, kernel_size=1)
        self.con3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

    def forward(self, features):
        x1, x2, x3 = features

        # 1. Прогоняем самый глубокий уровень через 1x1 свертку
        p3 = self.con3_1(x3)

        # 2. Поднимаем разрешение p3 и складываем с 1x1 проекцией x2
        p3_upsampled = F.interpolate(p3, scale_factor=2, mode='nearest')
        p2 = self.con2_1(x2) + p3_upsampled

        # 3. Поднимаем разрешение p2 и складываем с 1x1 проекцией x1
        p2_upsampled = F.interpolate(p2, scale_factor=2, mode='nearest')
        p1 = self.con1_1(x1) + p2_upsampled

        # 4. Финальные 3x3 свертки на каждом выходе шеи для устранения эффектов апсемплинга
        out3 = self.con3_3(p3)
        out2 = self.con2_3(p2)
        out1 = self.con1_3(p1)

        return (out1, out2, out3)

In [ ]:
a=  (torch.randn(1, 3, 28, 28),torch.randn(1, 3, 14, 14),torch.randn(1, 3, 7, 7))
t = Neck([3,3,3])
z = t(a)

In [ ]:
z[1].shape

torch.Size([1, 256, 14, 14])

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [ ]:
class Head(nn.Module):
    def __init__(self, in_f=256,num_c=4,f=3,anch=9,):
        super().__init__()
        self.cls_stems = nn.ModuleList()
        self.reg_stems = nn.ModuleList()

        self.cls_preds = nn.ModuleList()
        self.reg_preds = nn.ModuleList()

        for i in range(f):
          self.cls_stems.append(nn.Sequential(nn.Conv2d(in_f,in_f,padding=1,kernel_size=3),nn.ReLU()))
          self.reg_stems.append(nn.Sequential(nn.Conv2d(in_f,in_f,padding=1,kernel_size=3),nn.ReLU()))

          self.cls_preds.append(
                nn.Conv2d(in_f, anch * num_c, kernel_size=1)
            )
          self.reg_preds.append(
                nn.Conv2d(in_f, anch * 5, kernel_size=1)
            )
    def forward(self, neck_features):
        all_cls_outputs = []
        all_reg_outputs = []

        # Итерируемся по всем 3 картам признаков одновременно
        for i, x in enumerate(neck_features):
            # x — это ОДНА конкретная карта признаков на текущей итерации
            # (сначала out1, потом out2, потом out3)

            # Разделяем обработку этой карты на классификацию и регрессию
            cls_feat = self.cls_stems[i](x)
            reg_feat = self.reg_stems[i](x)

            # Получаем финальные предсказания для текущего масштаба
            cls_out = self.cls_preds[i](cls_feat)
            reg_out = self.reg_preds[i](reg_feat)

            all_cls_outputs.append(cls_out)
            all_reg_outputs.append(reg_out)

        # Возвращаем два списка: в каждом по 3 тензора предсказаний для каждого масштаба
        return all_cls_outputs, all_reg_outputs

In [ ]:
import torch
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.image_list import ImageList

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [ ]:
def decode_bboxes(a, anchor_centers,anchor_sizes):
  tx = a[:, :, 0]
  ty = a[:, :, 1]
  tw = a[:, :, 2]
  th = a[:, :, 3]

  center_x = anchor_centers[:, :, 0] + torch.sigmoid(tx) * anchor_sizes[:, :, 0]
  center_y = anchor_centers[:, :, 1] + torch.sigmoid(ty) * anchor_sizes[:, :, 1]

  w = torch.exp(tw) * anchor_sizes[:, :, 0]
  h = torch.exp(th) * anchor_sizes[:, :, 1]

  x_min = center_x - w / 2
  y_min = center_y - h / 2
  x_max = center_x + w / 2  # Добавили x_max
  y_max = center_y + h / 2  # Добавили y_max

  # Возвращаем xyxy
  return torch.stack([x_min, y_min, x_max, y_max], dim=-1)

In [ ]:
anchor_sizes = ((32,64,128),(32,64,128),(32,64,128))
aspect_ratios = ((0.5, 1.0, 2.0),(0.5, 1.0, 2.0),(0.5, 1.0, 2.0))
# Создаем генератор
generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)

# Имитируем батч из одного изображения 256x256
image_list = ImageList(torch.zeros(1, 3, 224, 224), [(224, 224)])

# Имитируем одну карту признаков, уменьшенную в 8 раз (stride = 8)
# Размер карты будет 32x32 (256 / 8)
feature_maps = [
  torch.randn(1, 256, 28, 28),  # Левел 1: 40x40 (stride=8)
  torch.randn(1, 256, 14, 14),  # Левел 2: 20x20 (stride=16)
  torch.randn(1, 256, 7, 7)   # Левел 3: 10x10 (stride=32)
]
anchors = generator(image_list, feature_maps)
anchors = torch.stack(anchors,dim=0)
anchors.shape

torch.Size([1, 9261, 4])

In [ ]:
class Detector(nn.Module):



    def __init__(self):
        super().__init__()
        self.num_classes=4
        self.backbone = Backbone()
        self.neck = Neck([128,256,512])
        self.head = Head(in_f=256,num_c=4)
        anchor_sizes = ((32,64,128),(32,64,128),(32,64,128))
        aspect_ratios = ((0.5, 1.0, 2.0),(0.5, 1.0, 2.0),(0.5, 1.0, 2.0))
        # Создаем генератор
        generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)

        # Имитируем батч из одного изображения 256x256
        image_list = ImageList(torch.zeros(1, 3, 224, 224), [(224, 224)])

        # Имитируем одну карту признаков, уменьшенную в 8 раз (stride = 8)
        # Размер карты будет 32x32 (256 / 8)
        feature_maps = [
          torch.randn(1, 256, 28, 28),  # Левел 1: 40x40 (stride=8)
          torch.randn(1, 256, 14, 14),  # Левел 2: 20x20 (stride=16)
          torch.randn(1, 256, 7, 7)   # Левел 3: 10x10 (stride=32)
        ]
        anchors = generator(image_list, feature_maps)
        anchors = torch.stack(anchors,dim=0)

        self.register_buffer("anchor_centers", (anchors[:, :, :2] + anchors[:, :, 2:]) / 2)
        self.register_buffer("anchor_sizes", (anchors[:, :, 2:] - anchors[:, :, :2]))
    def forward(self,x):
        c = self.backbone(x)
        f = self.neck(c)
        cls1,reg1 = self.head(f)
        a_cls = []
        a_reg = []
        a_conf = []
        N=x.shape[0]
        for i in range(3):
          cls = cls1[i]
          reg = reg1[i]
          cls_r = cls.permute(0,2,3,1).contiguous()
          cls_r = cls_r.view(N, -1, 4)

          reg_r = reg.permute(0,2,3,1).contiguous()
          reg_r = reg_r.view(N, -1, 5)

          # Создаем генератор
          conf_r = reg_r[:,:,4]
          reg_r = reg_r[:,:,:4]
          a_cls.append(cls_r)
          a_reg.append(reg_r)
          a_conf.append(conf_r)
        cls = torch.cat(a_cls,dim=1)
        reg = torch.cat(a_reg,dim=1)
        conf = torch.cat(a_conf, dim=1).squeeze(-1)
        reg_r = decode_bboxes(reg,self.anchor_centers,self.anchor_sizes)
        if self.training:
            return cls,reg_r,conf


        conf_r = torch.sigmoid(conf)
        cls_r = torch.softmax(cls,dim=2)
        return conf_r,reg_r,cls_r

In [ ]:


a=  torch.randn(1, 3, 224, 224) # Левел 1: 40x40 (stride=8)
r = Detector()
r(a)[0].shape

torch.Size([1, 9261, 4])

In [ ]:
a=  (torch.randn(1, 3, 224, 224))
r = Backbone()

r(a)


(tensor([[[[0.4307, 0.0903, 0.0000,  ..., 0.4436, 0.9982, 0.2714],
           [0.0000, 0.0000, 0.1235,  ..., 0.0000, 0.0000, 0.5221],
           [0.0342, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.4378, 0.0661, 0.2006,  ..., 0.0000, 0.7902, 1.0070],
           [0.5164, 0.2222, 0.0000,  ..., 0.0000, 0.1261, 0.0216],
           [0.0000, 0.0000, 0.2655,  ..., 0.0000, 0.0892, 0.0528]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0877, 0.4927],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.1717],
           [0.6692, 0.0000, 0.0000,  ..., 1.0284, 0.2019, 0.0000],
           ...,
           [0.2553, 0.1363, 0.1924,  ..., 0.1594, 0.5500, 0.4191],
           [0.8483, 0.2525, 0.1184,  ..., 0.5717, 1.2839, 0.3705],
           [0.1215, 0.0989, 0.2582,  ..., 0.0000, 0.0511, 0.0000]],
 
          [[0.0804, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.3633],
           [0.0000, 0.0056, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.00

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [ ]:
from torchvision.ops import box_iou

In [ ]:
def TAL_assigner(
    pred_scores: torch.Tensor,
    pred_bboxes: torch.Tensor,
    anchor_centers: torch.Tensor,
    gt_bboxes: torch.Tensor,
    gt_labels: torch.Tensor,
    topk: int = 13,
    alpha: float = 1.0,
    beta: float = 6.0
) -> torch.Tensor:
    """
    Реализация алгоритма Task Alignment Learning (TAL) для присвоения таргетов.

    Аргументы:
        pred_scores: Тензор вероятностей классов [num_anchors, num_classes] (после sigmoid/softmax).
        pred_bboxes: Тензор предсказанных рамок [num_anchors, 4] в формате (x_min, y_min, x_max, y_max).
        anchor_centers: Тензор координат центров якорей [num_anchors, 2] в формате (cx, cy).
        gt_bboxes: Тензор ground truth рамок [num_gt, 4] в формате (x_min, y_min, x_max, y_max).
        gt_labels: Тензор ground truth меток классов [num_gt].
        topk: Количество выбираемых якорей для каждого GT.
        alpha, beta: Гиперпараметры метрики выравнивания t.

    Возвращает:
        assigned_gt_inds: Тензор [num_anchors], содержащий индексы GT для каждого якоря.
                          Значение -1 означает, что якорь является фоном (background).
    """
    num_gt = gt_bboxes.size(0)
    num_anchors = pred_bboxes.size(0)

    # Если на изображении нет объектов (GT), все якоря становятся фоном (-1)
    if num_gt == 0:
        return torch.full((num_anchors,), -1, dtype=torch.long, device=pred_bboxes.device)

    # =========================================================================
    # 1. Посчитать значение метрики для каждого предсказанного ббокса
    # Формула: t = s^alpha * u^beta
    # =========================================================================

    # u: Считаем IoU между всеми GT и предсказаниями
    ious = box_iou(gt_bboxes, pred_bboxes)  # [num_gt, num_anchors]

    # s: Достаем вероятности только для целевых (GT) классов
    # pred_scores[:, gt_labels] дает размер [num_anchors, num_gt],
    # транспонируем (.t()) для получения [num_gt, num_anchors]
    s_for_gt = pred_scores[:, gt_labels].t()

    # t: Вычисляем саму метрику (Alignment Metric)
    t = (s_for_gt ** alpha) * (ious ** beta)  # [num_gt, num_anchors]

    # =========================================================================
    # 2. Отфильтровать предсказания на основе GT (центры внутри GT)
    # =========================================================================

    anchor_centers = anchor_centers.view(-1, 2)

    cx = anchor_centers[:, 0].unsqueeze(0)  # [1, num_anchors]
    cy = anchor_centers[:, 1].unsqueeze(0)  # [1, num_anchors]

    gt_xmin = gt_bboxes[:, 0].unsqueeze(1)
    # [num_gt, 1]
    gt_ymin = gt_bboxes[:, 1].unsqueeze(1)
    gt_xmax = gt_bboxes[:, 2].unsqueeze(1)
    gt_ymax = gt_bboxes[:, 3].unsqueeze(1)

    # Маска: [num_gt, num_anchors] -> True, если центр якоря попадает внутрь конкретного GT
    is_in_gts = (cx >= gt_xmin) & (cx <= gt_xmax) & (cy >= gt_ymin) & (cy <= gt_ymax)

    # Зануляем метрику `t` для всех якорей, которые не лежат внутри GT
    t = t * is_in_gts

    # =========================================================================
    # 3. Для каждого GT выбрать top-k самых подходящих предсказаний
    # =========================================================================

    # Если якорей меньше, чем запрошенный topk, берем их количество
    k = min(topk, num_anchors)

    # Находим индексы k максимальных значений метрики t для каждого GT
    _, topk_idxs = torch.topk(t, k, dim=1)  # [num_gt, k]

    # Создаем булеву маску для top-k
    is_in_topk = torch.zeros_like(t, dtype=torch.bool)
    is_in_topk.scatter_(1, topk_idxs, True)

    # Якорь является кандидатом, если он внутри GT И вошел в top-k
    candidate_mask = is_in_gts & is_in_topk  # [num_gt, num_anchors]

    # =========================================================================
    # 4. Разрешение конфликтов: если предсказание подходит для нескольких GT
    # Выбираем GT с наибольшим пересечением по IoU
    # =========================================================================

    # Оставляем значения IoU только для валидных кандидатов (остальное зануляем)
    valid_ious = ious * candidate_mask  # [num_gt, num_anchors]

    # Для каждого якоря находим максимальное IoU и индекс соответствующего GT
    max_iou, matched_gt_inds = valid_ious.max(dim=0)  # [num_anchors]

    # Индексы GT от 0 до num_gt - 1. Все якоря, у которых max_iou == 0
    # (то есть они не подошли ни одному GT), размечаем как фон (-1)
    assigned_gt_inds = matched_gt_inds.clone()
    assigned_gt_inds[max_iou == 0] = -1

    return assigned_gt_inds

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [ ]:
from torchvision.ops import distance_box_iou_loss

In [ ]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [ ]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [ ]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 0.9929903149604797


In [ ]:
def diou_loss(pred_boxes: torch.Tensor, gt_boxes: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    """
    Вычисляет DIoU loss между предсказанными и истинными ббоксами.
    Ожидаемый формат ббоксов: (x_1, y_1, x_2, y_2).
    """
    # 1. Находим площади обоих ббоксов
    area_pred = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (pred_boxes[:, 3] - pred_boxes[:, 1])
    area_gt = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])

    # 2. Считаем пересечение между ббоксами (Intersection)
    # Находим координаты углов пересечения
    lt_inter = torch.max(pred_boxes[:, :2], gt_boxes[:, :2])  # левый верхний угол [x1^I, y1^I]
    rb_inter = torch.min(pred_boxes[:, 2:], gt_boxes[:, 2:])  # правый нижний угол [x2^I, y2^I]

    # Ширина и высота пересечения (clamp(min=0) реализует условие "Иначе I=0")
    wh_inter = (rb_inter - lt_inter).clamp(min=0)
    inter = wh_inter[:, 0] * wh_inter[:, 1]  # I

    # 3. Считаем объединение (Union) и IoU
    union = area_pred + area_gt - inter
    iou = inter / (union + eps)  # добавляем eps для защиты от деления на 0

    # 4. Считаем диагональ выпуклой оболочки (c^2)
    lt_hull = torch.min(pred_boxes[:, :2], gt_boxes[:, :2])  # левый верхний угол оболочки [x1^c, y1^c]
    rb_hull = torch.max(pred_boxes[:, 2:], gt_boxes[:, 2:])  # правый нижний угол оболочки [x2^c, y2^c]

    wh_hull = (rb_hull - lt_hull).clamp(min=0)
    # c^2 = (x2^c - x1^c)^2 + (y2^c - y1^c)^2
    c2 = wh_hull[:, 0]**2 + wh_hull[:, 1]**2 + eps

    # 5. Рассчитываем расстояние между центрами ббоксов (d^2)
    center_pred = (pred_boxes[:, :2] + pred_boxes[:, 2:]) / 2
    center_gt = (gt_boxes[:, :2] + gt_boxes[:, 2:]) / 2

    # Евклидово расстояние в квадрате (по осям X и Y)
    d2 = ((center_pred - center_gt) ** 2).sum(dim=1)

    # 6. Собираем всё в итоговую формулу лосса
    diou = iou - (d2 / c2)
    loss = 1 - diou

    # Возвращаем усредненное значение (reduction="mean")
    return loss.mean()

In [ ]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

In [ ]:
class ComputeLoss:
    """ Базовый расчет лосса.

    Параметры
    ---------
    bbox_loss : Локализационная часть лосса
    obj_loss : Лосс для Confidence score
    cls_loss : Классификационная часть лосса
    weight_bbox, weight_obj, weight_cls : Константы для баллансировки частей лосса
    """
    def __init__(self,
            bbox_loss=None, obj_loss=None, cls_loss=None,
            weight_bbox=5, weight_obj=1, weight_cls=1
        ):
        self.bbox_loss = nn.SmoothL1Loss() if bbox_loss is None else bbox_loss
        self.obj_loss = nn.BCEWithLogitsLoss() if obj_loss is None else obj_loss
        self.cls_loss = nn.BCEWithLogitsLoss() if cls_loss is None else cls_loss
        self.weight_bbox = weight_bbox
        self.weight_obj = weight_obj
        self.weight_cls = weight_cls

    def __call__(self, predicts, targets):
        """ Расчет лосса для пары (предсказание, таргет)

        Параметры
        ---------
        predicts : Предсказания модели для одной картинки: Смещения, objectness score и логиты для классов
        targets : Gt значения для расчета лосса, а именно: GT смещения, GT objectness score и GT ohe классы
        """
        pred_offsets, pred_obj_logits, pred_cls_logits = predicts
        target_boxes, target_obj, target_cls = targets
        # Confidence score считается только для предсказаний соотв отрицательным и положительным якорям
        valid_mask = target_obj != -1
        loss_obj = self.obj_loss(pred_obj_logits[valid_mask], target_obj[valid_mask])

        # Локализационная и классификационные части считаются для предсказаинй соотв положительным якорям
        pos_mask = target_obj == 1
        if pos_mask.sum() > 0:
            loss_cls = self.cls_loss(pred_cls_logits[pos_mask], target_cls[pos_mask])
            loss_bbox = self.bbox_loss(pred_offsets[pos_mask], target_boxes[pos_mask])
        else:
            loss_cls = torch.tensor(0.0, device=pred_offsets.device)
            loss_bbox = torch.tensor(0.0, device=pred_offsets.device)
        return self.weight_bbox * loss_bbox + self.weight_obj * loss_obj + self.weight_cls * loss_cls

In [ ]:
class Runner:
    """ Базовый класс для обучения и валидации модели.

    Параметры
    ---------
    model : torch модель, которая будет обучаться.
    compute_loss : экземпляр класса ComputeLoss (или другого с реализованным методом __call__).
    optimizer : torch optimizer
    train_dataloader : torch dataloader семплирующий данные для обучения модели.
    assign_target_method : callable, который решает задачу сопоставления якорей и таргета (например, assign_target)
    deivce : девайс на котором будет происходить обучения, по дефолту "cpu"
    scheduler : torch scheduler
    assign_target_kwargs : доп параметры для функции в `assign_target_method`,
    val_dataloader : torch dataloader загружающий валидационные данные.
    score_threshold : При расчете метрики на валидации, все предсказания,
        с (confidence score * cls_probs) < score_threshold будут проигнорированны.
    nms_threshold : Предсказания, имеющие пересечение по IoU >= nms_threshold будут считаться одним предсказанием.
    max_boxes_per_cls : Максимальное количество ббоксов на изображение для одного класса после фильтрации по `score_threshold`.
    """
    def __init__(self, model, compute_loss, optimizer, train_dataloader, assign_target_method, device=None,
                 scheduler=None, assign_target_kwargs=None,
                 val_dataloader=None, val_every=5, score_threshold=0.1, nms_threshold=0.5, max_boxes_per_cls=8):
        self.model = model
        self.compute_loss = compute_loss
        self.optimizer = optimizer
        self.train_dataloader = train_dataloader
        assign_target_kwargs = {} if assign_target_kwargs is None else assign_target_kwargs
        self.assign_target_method = partial(assign_target_method, **assign_target_kwargs)
        self.device = "cpu" if device is None else device
        self.scheduler = scheduler

        # Валидационные параметры
        self.val_dataloader = val_dataloader
        self.val_every = val_every
        self.score_threshold = score_threshold
        self.nms_threshold = nms_threshold
        self.max_boxes_per_cls = max_boxes_per_cls

        # Вспомогательные массивы
        self.batch_loss = []
        self.epoch_loss = []
        self.val_metric = []

    def _run_train_epoch(self, dataloader, verbose=True):
        """ Обучить модель одну эпоху на данных из `dataloader` """
        self.model.train()
        batch_loss = []
        for images, targets in (pbar := tqdm(dataloader, desc=f"Process train epoch", leave=False)):
            images = images.to(self.device)

            # outputs = (cls, reg, conf) - то, что возвращает ваш Detector в режиме train
            outputs = self.model(images)
            pred_cls, pred_reg, pred_conf = outputs

            accum_loss = 0.0
            for ix in range(images.shape[0]):
                gt_boxes = targets[ix]['boxes'].to(self.device)
                gt_labels = targets[ix]['labels'].to(self.device)

                # 1. Формат GT: Переводим из (x, y, w, h) в (x_min, y_min, x_max, y_max)
                gt_xyxy = gt_boxes.clone()
                if gt_boxes.numel() > 0:
                    gt_xyxy[:, 2:] = gt_xyxy[:, :2] + gt_xyxy[:, 2:]

                # 2. Декодируем предсказания регрессии в координаты для TAL_assigner и DIoU.
                # Берем срез [ix:ix+1], чтобы сохранить размерность батча = 1 для функции decode.
                pred_bboxes = pred_reg[ix]
                pred_scores = torch.sigmoid(pred_cls[ix])

                # 3. Вызываем TAL_assigner
                assigned_gt_inds = self.assign_target_method(
                    pred_scores=pred_scores,
                    pred_bboxes=pred_bboxes,
                    anchor_centers=self.model.anchor_centers,
                    gt_bboxes=gt_xyxy,       # Передаем GT в формате углов
                    gt_labels=gt_labels
                )

                # 4. Формируем таргеты в том виде, в котором их ждет ComputeLoss
                num_anchors = pred_bboxes.shape[0]
                target_boxes = torch.zeros_like(pred_bboxes)
                target_obj = torch.zeros(num_anchors, device=self.device)
                target_cls = torch.zeros((num_anchors, self.model.num_classes), device=self.device)

                if gt_xyxy.numel() > 0:
                    pos_mask = assigned_gt_inds >= 0
                    if pos_mask.sum() > 0:
                        target_obj[pos_mask] = 1
                        matched_gts = assigned_gt_inds[pos_mask]
                        # Сохраняем реальные GT ббоксы, а не смещения (оффсеты), так как DIoU нужны ббоксы
                        target_boxes[pos_mask] = gt_xyxy[matched_gts]
                        target_cls[pos_mask, gt_labels[matched_gts]] = 1

                # 5. Считаем лосс.
                # ВАЖНО: передаем декодированные pred_bboxes вместо сырых предсказаний,
                # чтобы DIoU loss внутри считался по реальным координатам!
                predicts_ix = (pred_bboxes, pred_conf[ix], pred_cls[ix])
                targets_ix = (target_boxes, target_obj, target_cls)

                loss = self.compute_loss(predicts_ix, targets_ix)
                accum_loss += loss

            accum_loss = accum_loss / images.shape[0]
            batch_loss.append(accum_loss.cpu().detach().item())

            # Делаем шаг оптимизатора после расчета лосса для всех элементов батча
            self.optimizer.zero_grad()
            accum_loss.backward()
            self.optimizer.step()

            # Обновляем описание tqdm бара
            if verbose:
                pbar.set_description(f"Current batch loss: {batch_loss[-1]:.4}")

        return batch_loss

    def train(self, num_epochs=10, verbose=True):
        """ Обучаем модель заданное количество эпох. """
        val_desc = ""
        for epoch in (epoch_pbar := tqdm(range(1, num_epochs+1), desc="Train epoch", total=num_epochs)):
            # Обучаем модель одну эпоху
            loss = self._run_train_epoch(self.train_dataloader, verbose=verbose)
            self.batch_loss.extend(loss)
            self.epoch_loss.append(np.mean(self.batch_loss[-len(self.train_dataloader):]))

            # Делаем валидацию, если был передан валидационный датасет
            if self.val_dataloader is not None and epoch % self.val_every == 0:
                val_metric = self.validate()
                self.val_metric.append(val_metric)
                val_desc = f" Val {val_metric:.4}"

            # Обновляем описание tqdm бара усредненным значением лосса за предыдую эпоху
            if verbose:
                epoch_pbar.set_description(f"Last epoch loss: Train {self.epoch_loss[-1]:.4}" + val_desc)
            # Делаем шаг scheduler'a если он был передан
            if self.scheduler is not None:
                self.scheduler.step()

    @torch.no_grad()
    def validate(self, dataloader=None):
        """ Метод для валидации модели. Если dataloader не передан, будет использоваться self.val_dataloder.
        Возвращает mAP (0.5 ... 0.95).
        """
        self.model.eval()
        dataloader = self.val_dataloader if dataloader is None else dataloader
        # Считаем метрику mAP с помощью функции из torchmetrics
        metric = MeanAveragePrecision(box_format="xywh", iou_type="bbox")
        for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
            images = images.to(self.device)
            outputs = self.model(images)
            predicts = _filter_predictions(outputs, self.score_threshold, self.nms_threshold,
                                           max_boxes_per_cls=self.max_boxes_per_cls, return_type="torch")
            metric.update(predicts, targets)
        return metric.compute()["map"].item()

    def plot_loss(self, row_figsize=3):
        nrows = 2 if self.val_metric else 1
        _, ax = plt.subplots(nrows, 1, figsize=(12, row_figsize*nrows), tight_layout=True)
        ax = np.array([ax]) if not isinstance(ax, np.ndarray) else ax
        ax[0].plot(self.batch_loss, label="Train batch Loss", color="tab:blue")
        ax[0].plot(np.arange(1, len(self.batch_loss)+1, len(self.train_dataloader)), self.epoch_loss,
                   color="tab:orange", label="Train epoch Loss")
        ax[0].grid()
        ax[0].set_title("Train Loss")
        ax[0].set_xlabel("Number of Iterations")
        ax[0].set_ylabel("Loss")
        if self.val_metric:
            ax[1].plot(np.arange(self.val_every, len(self.batch_loss)+1, len(self.val_dataloader) * self.val_every),
                       np.array(self.val_metric) * 100, color="tab:green", label="Validation mAP")
            ax[1].grid()
            ax[1].set_title("Valiation mAP")
            ax[1].set_xlabel("Number of Iterations")
            ax[1].set_ylabel("mAP (%)")
        plt.legend()
        plt.show()

def _filter_predictions(predictions, score_threshold=0.1, nms_threshold=0.5, max_boxes_per_cls=8, return_type="list"):
    """ Ббоксы в `predictions` должны быть в формате (x_min, y_min, w, h). """
    # Итоговый скор считается как произведение уверенности модели в том что в данном якоре
    # и вероятность каждого класса в данном якоре.
    bboxes, confidences, cls_probs = predictions
    all_final_scores = confidences[:, :, None] * cls_probs

    num_classes = cls_probs.shape[-1]
    final_predictions = []
    # Для каждого элемента в `predictions` независимо выбираем ббоксы и скоры
    for boxes, final_scores in zip(bboxes, all_final_scores):
        preds = {"boxes": [], "labels": [], "scores": []}

        # Для каждого класса отдельно фильтруем ббоксы с помощью NMS
        for cls in range(num_classes):
            cls_scores = final_scores[:, cls]
            # Фильтруем ббоксы, score которых меньше порога
            keep_ixs = cls_scores > score_threshold
            if keep_ixs.sum() == 0:
                continue
            cls_boxes = boxes[keep_ixs]
            cls_scores = cls_scores[keep_ixs]

            # Если предсказаний слишком много, выбираем только самые уверенные
            if len(cls_boxes) > max_boxes_per_cls:
                pos = torch.argsort(cls_scores, descending=True)
                cls_boxes = cls_boxes[pos[:max_boxes_per_cls]]
                cls_scores = cls_scores[pos[:max_boxes_per_cls]]

            # Конвертируем ббоксы в формат x_min, y_min, x_max, y_max
            boxes_xyxy = cls_boxes.clone()
            boxes_xyxy[:, 2:] = boxes_xyxy[:, :2] + boxes_xyxy[:, 2:]
            # Запускаем NMS по всем оставшимся ббоксам класса cls
            pred_ixs = nms(boxes_xyxy, cls_scores, nms_threshold)
            # Сохраняем все предсказания для класса cls
            for ix in pred_ixs:
                preds["boxes"].append(cls_boxes[ix].cpu().tolist())
                preds["labels"].append(cls)
                preds["scores"].append(cls_scores[ix].item())
        if return_type == "torch":
            for key, item in preds.items():
                preds[key] = torch.tensor(item)
        elif return_type != "list":
            raise ValueError(f"Received unexpected `return_type`. Could be either `torch` or `list`, not {return_type}")
        final_predictions.append(preds)
    return final_predictions

In [ ]:
import torch.optim

In [ ]:
model = Detector()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
cmpl = ComputeLoss()

In [ ]:
# Создаем инстанс лосса, указывая diou_loss для локализации
loss_fn = ComputeLoss(
    bbox_loss=diou_loss,  # Ваша функция DIoU
    # Остальные веса и функции можно оставить по дефолту
)

# Создаем Runner, указывая TAL_assigner
runner = Runner(
    model=model,
    compute_loss=loss_fn,
    optimizer=optimizer,
    train_dataloader=train_dataloader,
    assign_target_method=TAL_assigner, # <--- Передаем новую функцию
    # assign_target_kwargs={'topk': 13} # если нужно передать гиперпараметры
)

In [ ]:
t = Runner(model,loss_fn,optimizer,train_dataloader,TAL_assigner)
t.train(num_epochs=10)

Train epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Process train epoch:   0%|          | 0/8 [00:00<?, ?it/s]

Process train epoch:   0%|          | 0/8 [00:00<?, ?it/s]

Process train epoch:   0%|          | 0/8 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [ ]:

# Место для обучения, валидации и экспериментов!


Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [ ]:
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
    """ Метод для валидации модели.
    Возвращает mAP (0.5 ... 0.95).
    """
    self.model.eval()
    # Считаем метрику mAP с помощью функции из torchmetrics
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = self.model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
        metric.update(predicts, targets)
    return metric.compute()["map"].item()
